# Background Estimation Overview
The lepton background consists of 3 main parts: 
- The lepton must fail to be reconstructed as a lepton (PVeto)
- If a lepton fails to be reconstructed, probability of a single lepton event to pass our offline requirements of $p_{T}^{miss, no \mu} > 120 GeV$ and $|\Delta \Phi(leading jet, p_{T}^{miss, no \mu})| > 0.5$
- The single lepton event must pass the $p_{T}^{miss}$ trigger given that the lepton is not reconstructed as such and the event passes the offline requirements above

# PVeto
We use tag and probe studies to estimate PVeto. The cuts used to generate the tag and probe are shown below for the electrons:
![Electron Tag and Probe Selections](../Assets/ElectronTagAndProbeSelections.png)

The selections for the muons are below: 

![Muon Tag and Probe Selections](../Assets/MuonTagAndProbeSelections.png)


In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import awkward as ak
import ROOT
import vector
vector.register_awkward()
%jsroot on


In [2]:
def print_cutflow(arrays, label):
    n_events = len(arrays["trk_pt"])
    n_tracks = ak.sum(ak.num(arrays["trk_pt"]))
    print(f"{label:50s}  events: {n_events:>8,}  tracks: {n_tracks:>8,}")


In [3]:
def dR_mask(arrays, obj_prefix, trk_prefix="trk", dr_cut=0.5, pt_cut=None, mode="all"):
    tracks = ak.zip({
        "eta": arrays[f"{trk_prefix}_eta"],
        "phi": arrays[f"{trk_prefix}_phi"]
    })
    objs = ak.zip({
        "eta": arrays[f"{obj_prefix}_eta"],
        "phi": arrays[f"{obj_prefix}_phi"],
        "pt":  arrays[f"{obj_prefix}_pt"],
    })
    
    if pt_cut is not None:
        objs = objs[objs.pt > pt_cut]
    
    t, o  = ak.unzip(ak.cartesian([tracks, objs], nested=True))
    dphi  = np.arctan2(np.sin(t.phi - o.phi), np.cos(t.phi - o.phi))
    dR    = np.sqrt((t.eta - o.eta)**2 + dphi**2)
    
    if mode == "all":
        return ak.all(dR > dr_cut, axis=2)  # far from ALL objects
    elif mode == "any":
        return ak.any(dR > dr_cut, axis=2)  # far from AT LEAST ONE object

In [4]:
NTUPLE_FILE = "root://cmseosmgm01.fnal.gov:1094//store/group/lpclonglived/DisappTrks/Muon0/2024_C_v1_Muon0/260423_221458/0000/ntuple_9.root"
TREE_NAME   = "ntuplizer/Events"   # adjust to match treeName + TFileService path

In [5]:
f    = uproot.open(NTUPLE_FILE)
tree = f[TREE_NAME]
print(f"Entries: {tree.num_entries:,}\n")
print("Branches:")
for b in tree.keys():
    print(f"  {b}")

arrays = tree.arrays(["muon_pt", "muon_eta", "muon_phi", "muon_charge", "ele_pt", "ele_eta", "ele_phi", "tau_pt", "tau_eta", "tau_phi", "trk_pt", "trk_eta","trk_hp_trackerLayersWithMeasurement", "trk_hp_trackerLayersWithoutMeasurement_OUTER", "trk_hp_trackerLayersWithMeasurement","trk_relativePFIso", "trk_phi", "trk_eta", "trk_caloTotal", "trk_charge", "jet_pt", "jet_phi", "jet_eta"], entry_stop=5_000_000, library="ak")

Entries: 2,745,325

Branches:
  run
  lumi
  eventNum
  met_pt
  met_phi
  metNoMu_pt
  metNoMu_phi
  trk_pt
  trk_eta
  trk_phi
  trk_dxy
  trk_dxyError
  trk_dz
  trk_dzError
  trk_charge
  trk_fromPV
  trk_deltaEta
  trk_deltaPhi
  trk_isHighPurityTrack
  trk_isTightTrack
  trk_isLooseTrack
  trk_dEdxStrip
  trk_dEdxPixel
  trk_caloEm
  trk_caloHad
  trk_caloTotal
  trk_crossedEcalStatus
  trk_crossedHcalStatus
  trk_pfIso
  trk_relativePFIso
  trk_pfIso_chHad
  trk_pfIso_neutHad
  trk_pfIso_photon
  trk_pfIso_puChHad
  trk_miniIso_chHad
  trk_miniIso_neutHad
  trk_miniIso_photon
  trk_miniIso_puChHad
  trk_miniIso_relative
  trk_dPhiMet
  trk_dPhiMetNoMu
  trk_ptOverMetNoMu
  trk_hp_numberOfAllHits
  trk_hp_numberOfAllTrackerHits
  trk_hp_numberOfValidHits
  trk_hp_numberOfValidTrackerHits
  trk_hp_numberOfValidPixelHits
  trk_hp_numberOfValidPixelBarrelHits
  trk_hp_numberOfValidPixelEndcapHits
  trk_hp_numberOfValidStripHits
  trk_hp_numberOfValidStripTIBHits
  trk_hp_numberOfVal

In [6]:
print_cutflow(arrays, "Initial")

Initial                                             events: 2,745,325  tracks: 1,406,839


In [7]:
# ── 1. Muon event filter ─────────────────────────────────────────────────
muon_pt_cut  = arrays["muon_pt"] > 26
muon_eta_cut = np.abs(arrays["muon_eta"]) < 2.1
good_muons   = muon_pt_cut & muon_eta_cut
arrays       = arrays[ak.any(good_muons, axis=1)]
good_muons = good_muons[ak.any(good_muons, axis=1)]  # keep in sync
# then filter muon branches to only keep good muons
arrays["muon_pt"]     = arrays["muon_pt"][good_muons]
arrays["muon_eta"]    = arrays["muon_eta"][good_muons]
arrays["muon_phi"]    = arrays["muon_phi"][good_muons]
arrays["muon_charge"] = arrays["muon_charge"][good_muons]
print_cutflow(arrays, "After muon event filter (pT>26, |eta|<2.1)")

After muon event filter (pT>26, |eta|<2.1)          events: 2,070,899  tracks: 1,114,146


In [8]:
# ── 2. Track pT , ecalo, and eta cuts ─────────────────────────────────────────────
track_pt_cut    = arrays["trk_pt"] > 30
track_eta_cut   = np.abs(arrays["trk_eta"]) < 2.1
track_eta_cut_1 = (np.abs(arrays["trk_eta"]) < 0.15) | (np.abs(arrays["trk_eta"]) > 0.35)
track_eta_cut_2 = (np.abs(arrays["trk_eta"]) < 1.42) | (np.abs(arrays["trk_eta"]) > 1.65)
track_eta_cut_3 = (np.abs(arrays["trk_eta"]) < 1.55) | (np.abs(arrays["trk_eta"]) > 1.85)
track_ecalo_cut = arrays["trk_caloTotal"] < 10 
track_mask      = track_pt_cut & track_eta_cut & track_eta_cut_1 & track_eta_cut_2 & track_eta_cut_3 & track_ecalo_cut

# drop events with no passing tracks
event_filter = ak.any(track_mask, axis=1)
arrays       = arrays[event_filter]
track_mask   = track_mask[event_filter]

# filter all trk_ branches to only keep good tracks
for field in arrays.fields:
    if field.startswith("trk_"):
        arrays = ak.with_field(arrays, arrays[field][track_mask], field)

print_cutflow(arrays, "After track pT and eta cuts")

After track pT and eta cuts                         events:  481,234  tracks:  490,541


In [9]:

# get per-track boolean - DON'T slice arrays with this directly
jet_veto = dR_mask(arrays, "jet", dr_cut=0.5, pt_cut=None, mode="any")

# only drop events at the event level
event_filter = ak.any(track_mask, axis=1)
arrays       = arrays[event_filter]
jet_veto   = jet_veto[event_filter]
for field in arrays.fields:
    if field.startswith("trk_"):
        arrays = ak.with_field(arrays, arrays[field][jet_veto], field)

print_cutflow(arrays, "After jet dR veto (dR > 0.5)")

After jet dR veto (dR > 0.5)                        events:  481,234  tracks:  442,167


In [10]:
tracks = ak.zip({
    "pt":  arrays["trk_pt"],
    "eta": arrays["trk_eta"],
    "phi": arrays["trk_phi"],
})
electrons = ak.zip({
    "pt": arrays["ele_pt"], 
    "eta": arrays["ele_eta"],
    "phi": arrays["ele_phi"]
})
# cross join
tracks_paired, electrons_paired = ak.unzip(
    ak.cartesian([tracks, electrons], nested=True)
)
# compute dR
deta = tracks_paired.eta - electrons_paired.eta
dphi = np.arctan2(
    np.sin(tracks_paired.phi - electrons_paired.phi),
    np.cos(tracks_paired.phi - electrons_paired.phi)
)
dR = np.sqrt(deta**2 + dphi**2)

# mask: track must be dR > 0.15 from ALL electrons
electron_veto_mask = ak.all(dR > 0.15, axis=2)


# drop events with no passing tracks
event_filter = ak.any(electron_veto_mask, axis=1)
arrays       = arrays[event_filter]
electron_veto_mask   = electron_veto_mask[event_filter]  # sync the mask!


# filter all trk_ branches to only keep good tracks
for field in arrays.fields:
    if field.startswith("trk_"):
        arrays = ak.with_field(arrays, arrays[field][electron_veto_mask], field)


In [11]:
tracks = ak.zip({
    "pt":  arrays["trk_pt"],
    "eta": arrays["trk_eta"],
    "phi": arrays["trk_phi"],
})
taus = ak.zip({
    "pt": arrays["tau_pt"], 
    "eta": arrays["tau_eta"],
    "phi": arrays["tau_phi"]
})
# cross join
tracks_paired, taus_paired = ak.unzip(
    ak.cartesian([tracks, taus], nested=True)
)
# compute dR
deta = tracks_paired.eta - taus_paired.eta
dphi = np.arctan2(
    np.sin(tracks_paired.phi - taus_paired.phi),
    np.cos(tracks_paired.phi - taus_paired.phi)
)
dR = np.sqrt(deta**2 + dphi**2)

# mask: track must be dR > 0.15 from ALL electrons
tau_veto_mask = ak.all(dR > 0.15, axis=2)


# drop events with no passing tracks
event_filter = ak.any(tau_veto_mask, axis=1)
arrays       = arrays[event_filter]
tau_veto_mask   = tau_veto_mask[event_filter]  # sync the mask!


# filter all trk_ branches to only keep good tracks
for field in arrays.fields:
    if field.startswith("trk_"):
        arrays = ak.with_field(arrays, arrays[field][tau_veto_mask], field)


At this point we have a collection of quality tracks and quality muons we can now do the track-muon pair cuts to see if these a pair came from the Z boson

In [12]:
# build 4-vector records
tracks = ak.zip({
    "pt":               arrays["trk_pt"],
    "eta":              arrays["trk_eta"],
    "phi":              arrays["trk_phi"],
    "mass":             ak.ones_like(arrays["trk_pt"]) * 0.106,
    "charge":           arrays["trk_charge"],
    "missingOuterHits": arrays["trk_hp_trackerLayersWithoutMeasurement_OUTER"],
    "nLayers":          arrays["trk_hp_trackerLayersWithMeasurement"]
}, with_name="Momentum4D")

muons = ak.zip({
    "pt":     arrays["muon_pt"],
    "eta":    arrays["muon_eta"],
    "phi":    arrays["muon_phi"],
    "mass":   ak.ones_like(arrays["muon_pt"]) * 0.106,
    "charge": arrays["muon_charge"],
}, with_name="Momentum4D")

# ── Compute dR from each track to its closest muon in the event ───────────────
# shape: [event, track, muon]
tracks_all, muons_all = ak.unzip(ak.cartesian([tracks, muons], nested=True))
dphi_all   = np.arctan2(
    np.sin(tracks_all.phi  - muons_all.phi),
    np.cos(tracks_all.phi  - muons_all.phi)
)
dR_all          = np.sqrt((tracks_all.eta - muons_all.eta)**2 + dphi_all**2)
min_dR_to_muon  = ak.min(dR_all, axis=2)   # shape: [event, track]
dr_cut_pertrack = min_dR_to_muon > 0.15    # True if track is far from ALL muons

# ── Pair each track with each muon for invariant mass ────────────────────────
tracks_paired, muons_paired = tracks_all, muons_all   # reuse cartesian from above

pairs    = tracks_paired + muons_paired
inv_mass = pairs.mass   # shape: [event, track, muon]

dphi_paired = dphi_all  # already computed above
dR_paired   = dR_all

# ── Define cuts ───────────────────────────────────────────────────────────────
z_window      = (inv_mass > 81.2) & (inv_mass < 101.2)
opposite_sign = tracks_paired.charge * muons_paired.charge <= -1
same_sign     = tracks_paired.charge * muons_paired.charge >= 1
nLayers_cut   = tracks_paired.nLayers >= 4

# broadcast per-track dr_cut to [event, track, muon] for use in pair selections
dr_cut_broadcast = ak.broadcast_arrays(dr_cut_pertrack[:, :, np.newaxis], inv_mass)[0]

# ── Denominator: Z pairs (use ak.any over muon axis to count tracks, not pairs)
z_pairs_OS_pertrack = ak.any(z_window & opposite_sign & nLayers_cut, axis=2)  # [event, track]
z_pairs_SS_pertrack = ak.any(z_window & same_sign     & nLayers_cut, axis=2)

total_z_pairs_OS = ak.sum(ak.sum(z_pairs_OS_pertrack, axis=1))
total_z_pairs_SS = ak.sum(ak.sum(z_pairs_SS_pertrack, axis=1))
print(f"N_TP opposite charge: {total_z_pairs_OS:,}")
print(f"N_TP same charge:     {total_z_pairs_SS:,}")

# ── Numerator: probes passing veto ───────────────────────────────────────────
missing_outer_cut = tracks_paired.missingOuterHits >= 3   # [event, track, muon]

# dr_cut is now per-track (closest muon), broadcast to pair shape
tag_probe_OS_pertrack = ak.any(
    dr_cut_broadcast & missing_outer_cut & opposite_sign & z_window & nLayers_cut,
    axis=2
)
tag_probe_SS_pertrack = ak.any(
    dr_cut_broadcast & missing_outer_cut & same_sign     & z_window & nLayers_cut,
    axis=2
)

total_tag_probe_veto_OS = ak.sum(ak.sum(tag_probe_OS_pertrack, axis=1))
total_tag_probe_veto_SS = ak.sum(ak.sum(tag_probe_SS_pertrack, axis=1))
print(f"N_TP VETO:    {total_tag_probe_veto_OS:,}")
print(f"N_TP VETO SS: {total_tag_probe_veto_SS:,}")

PVeto = (total_tag_probe_veto_OS - total_tag_probe_veto_SS) / (total_z_pairs_OS - total_z_pairs_SS)
print(f"PVeto = {PVeto:.6f}")



N_TP opposite charge: 58,278
N_TP same charge:     26
N_TP VETO:    5
N_TP VETO SS: 5
PVeto = 0.000000


We can now calculate $P_{Veto}$

In [13]:
PVeto = (total_tag_probe_veto_OS - total_tag_probe_veto_SS) / (total_z_pairs_OS - total_z_pairs_SS)
PVeto

np.float64(0.0)

For muons this is what we are expecting: 
- total_tag_probe_veto_OS: 74
- total_tag_probe_veto_SS: 76
- total_z_pairs_OS: 2195289
- total_z_pairs_SS: 191

These numbers are what we previously had for 2023D with an integrated luminosity of 9.68 fb-1 